In [1]:
# Standard library imports
import sys
from pathlib import Path

# Third-party imports
import duckdb

# Add src to path
sys.path.insert(0, str(Path.cwd().parent / 'src'))

## 1. Feature Engineering with DuckDB

**Goal:** Create new features for analysis (Temporal, Location, Severity) using efficient SQL queries.
**Engine:** DuckDB (Out-of-core processing)

In [2]:
# Define paths
input_path = Path('../data/processed/integrated_clean.parquet')
output_path = Path('../data/processed/dashboard_data.parquet')

if not input_path.exists():
    raise FileNotFoundError(
        "Clean integrated dataset not found! Please run notebook 05_post_integration_cleaning.ipynb first."
    )

print("Initializing DuckDB...")
con = duckdb.connect(database=':memory:')

# Check schema
print("Inspecting schema...")
con.execute(f"CREATE OR REPLACE VIEW source_data AS SELECT * FROM '{input_path}'")
columns_info = con.execute("DESCRIBE source_data").fetchall()
columns = [col[0] for col in columns_info]
print(f"✓ Loaded source view with {len(columns)} columns")

Initializing DuckDB...
Inspecting schema...
✓ Loaded source view with 25 columns


## 2. Define Feature Logic (SQL)

We will construct a comprehensive SQL query to generate all features in one pass.

In [ ]:
# Helper to check if column exists
def get_col(possible_names, default='NULL'):
    for name in possible_names:
        if name in columns:
            return f'"{name}"'
    return default

# Identify column names dynamically
date_col = get_col(['CRASH DATE', 'CRASH_DATE'])
time_col = get_col(['CRASH TIME', 'CRASH_TIME'])
borough_col = get_col(['BOROUGH'])
lat_col = get_col(['LATITUDE', 'LATITUDE_crash'])
lon_col = get_col(['LONGITUDE', 'LONGITUDE_crash'])

injured_col = get_col(['NUMBER OF PERSONS INJURED', 'NUMBER_OF_PERSONS_INJURED'], '0')
killed_col = get_col(['NUMBER OF PERSONS KILLED', 'NUMBER_OF_PERSONS_KILLED'], '0')

ped_injured = get_col(['NUMBER OF PEDESTRIANS INJURED'], '0')
ped_killed = get_col(['NUMBER OF PEDESTRIANS KILLED'], '0')
cyc_injured = get_col(['NUMBER OF CYCLIST INJURED'], '0')
cyc_killed = get_col(['NUMBER OF CYCLIST KILLED'], '0')
mot_injured = get_col(['NUMBER OF MOTORIST INJURED'], '0')
mot_killed = get_col(['NUMBER OF MOTORIST KILLED'], '0')

print("Constructing feature engineering query...")

# Exclude crash-level aggregate columns to prevent double-counting
# These columns (NUMBER OF PERSONS INJURED/KILLED, etc.) are crash-level aggregates
# but our dataset is person-level. Summing them across person rows multiplies counts.
exclude_cols = [
    'NUMBER OF PERSONS INJURED', 'NUMBER OF PERSONS KILLED',
    'NUMBER OF PEDESTRIANS INJURED', 'NUMBER OF PEDESTRIANS KILLED', 
    'NUMBER OF CYCLIST INJURED', 'NUMBER OF CYCLIST KILLED',
    'NUMBER OF MOTORIST INJURED', 'NUMBER OF MOTORIST KILLED'
]
select_cols = [f'"{c}"' for c in columns if c not in exclude_cols]
select_clause = ', '.join(select_cols)

query = f"""
SELECT 
    {select_clause},
    -- Temporal Features
    YEAR({date_col}) as crash_year,
    MONTH({date_col}) as crash_month,
    DAY({date_col}) as crash_day,
    DAYOFWEEK({date_col}) as crash_day_of_week,
    DAYNAME({date_col}) as crash_day_name,
    QUARTER({date_col}) as crash_quarter,
    WEEK({date_col}) as crash_week_of_year,
    CASE 
        WHEN {time_col} IS NOT NULL THEN CAST(STRPTIME({time_col}, '%H:%M') AS TIME)
        ELSE NULL 
    END as crash_time_obj,
    CASE 
        WHEN {time_col} IS NOT NULL THEN EXTRACT(HOUR FROM CAST(STRPTIME({time_col}, '%H:%M') AS TIME))
        ELSE NULL 
    END as crash_hour,
    CASE 
        WHEN MONTH({date_col}) IN (12, 1, 2) THEN 'Winter'
        WHEN MONTH({date_col}) IN (3, 4, 5) THEN 'Spring'
        WHEN MONTH({date_col}) IN (6, 7, 8) THEN 'Summer'
        ELSE 'Fall'
    END as season,

    -- Location Features
    CASE 
        WHEN {borough_col} IS NOT NULL THEN TRIM(UPPER(SUBSTRING({borough_col}, 1, 1)) || LOWER(SUBSTRING({borough_col}, 2)))
        ELSE NULL 
    END as borough_clean,
    CASE WHEN {borough_col} IS NOT NULL THEN TRUE ELSE FALSE END as has_borough,
    CASE WHEN {lat_col} IS NOT NULL AND {lon_col} IS NOT NULL THEN TRUE ELSE FALSE END as has_coordinates,
    
    -- Severity Score (Person-Level)
    -- Using PERSON_INJURY column to calculate per-person severity
    CASE 
        WHEN PERSON_INJURY = 'Killed' THEN 10
        WHEN PERSON_INJURY = 'Injured' THEN 1
        ELSE 0
    END as severity_score

FROM source_data
"""

print("✓ Query constructed")

Constructing feature engineering query...
✓ Query constructed


### Severity Score Formula

The severity score uses a **1:10 injury-to-fatality weighting ratio** based on:

- **NHTSA Economic Analysis**: The economic value of a fatality (~$9.6M) is approximately 10 times higher than an injury ($57k-$155k average), giving a ratio of ~1:9.6, rounded to 1:10 for simplicity.
- **WHO Global Burden of Disease**: Disability-Adjusted Life Years (DALYs) show that fatalities have approximately 10x the impact of injuries.
- **Traffic Safety Standards**: This 1:10 ratio is commonly used in traffic safety analysis for severity classification.

**Formula**: `severity_score = (injuries × 1) + (deaths × 10)`

**Categories**:
- **No Injury**: score = 0 (property damage only)
- **Minor**: score 1-2 (minor injuries)
- **Moderate**: score 3-9 (multiple injuries)
- **Severe**: score ≥10 (fatalities or major incidents)

**Sources**:
- NHTSA: https://crashstats.nhtsa.dot.gov/Api/Public/ViewPublication/812013
- WHO: https://www.who.int/publications/i/item/9789241565684

## 3. Execute and Save

Stream the results directly to the final Parquet file.

In [4]:
print("Executing feature engineering and saving...")
output_path.parent.mkdir(parents=True, exist_ok=True)

copy_query = f"COPY ({query}) TO '{output_path}' (FORMAT PARQUET, COMPRESSION 'SNAPPY')"
con.execute(copy_query)

print("✅ DASHBOARD DATASET SAVED!")
print("="*60)
print(f"Path: {output_path}")
print(f"Size: {output_path.stat().st_size / (1024**2):.2f} MB")

# Verify
count = con.execute(f"SELECT COUNT(*) FROM '{output_path}'").fetchone()[0]
print(f"Records: {count:,}")

con.close()

Executing feature engineering and saving...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✅ DASHBOARD DATASET SAVED!
Path: ..\data\processed\dashboard_data.parquet
Size: 180.14 MB
Records: 5,376,331
